In [1]:
!pip install imagehash
import os
import shutil
from PIL import Image
import imagehash
from tqdm import tqdm

# 1. Define Paths
input_base = '/kaggle/input/datasets/tejpaviraj/arecanut'
input_train = os.path.join(input_base, 'Arecanut_dataset/Arecanut_dataset/train')
input_test = os.path.join(input_base, 'Arecanut_dataset/Arecanut_dataset/test')

# Kaggle Output (Writable)
output_base = '/kaggle/working/clean_arecanut'
os.makedirs(output_base, exist_ok=True)

# Global tracker for unique fingerprints
seen_hashes = set()

def process_and_deduplicate(source_dir, dest_dir, set_name):
    if not os.path.exists(source_dir):
        print(f"⚠️ Directory not found, skipping: {source_dir}")
        return

    print(f"\nScanning and cleaning {set_name} set...")
    os.makedirs(dest_dir, exist_ok=True)
    
    total_images = 0
    copied_images = 0
    duplicates = 0

    # Walk through every class folder (healthy, rot, etc.)
    for class_name in os.listdir(source_dir):
        class_source_path = os.path.join(source_dir, class_name)
        if not os.path.isdir(class_source_path): continue
            
        class_dest_path = os.path.join(dest_dir, class_name)
        os.makedirs(class_dest_path, exist_ok=True)

        for img_name in tqdm(os.listdir(class_source_path), desc=f"Class: {class_name}"):
            img_source_path = os.path.join(class_source_path, img_name)
            total_images += 1
            
            try:
                # Open image and calculate its Perceptual Hash (fingerprint)
                with Image.open(img_source_path) as img:
                    # phash looks at the visual features, so even slightly resized images are caught
                    img_hash = imagehash.phash(img) 
                
                # If we have NEVER seen this fingerprint before...
                if img_hash not in seen_hashes:
                    seen_hashes.add(img_hash)
                    shutil.copy2(img_source_path, os.path.join(class_dest_path, img_name))
                    copied_images += 1
                else:
                    duplicates += 1
            except Exception as e:
                print(f"Error reading {img_name}: {e}")

    print(f"✅ {set_name} Cleaned! Total: {total_images} | Kept: {copied_images} | Duplicates Trashed: {duplicates}")

# --- 2. Execute the Pipeline ---
# STRICT ORDER: Process Test sets FIRST to secure the "Exam" questions.
# If a duplicate is found later in Train, it will be thrown away.
process_and_deduplicate(input_test, os.path.join(output_base, 'test'), "VALIDATION (TEST)")
process_and_deduplicate(input_train, os.path.join(output_base, 'train'), "TRAIN")

print(f"\n🎉 Data Cleaning Complete! Your pure, un-leaked dataset is ready at: {output_base}")


Scanning and cleaning VALIDATION (TEST) set...


Class: Healthy_Nut: 100%|██████████| 472/472 [00:10<00:00, 46.20it/s]


✅ VALIDATION (TEST) Cleaned! Total: 2216 | Kept: 2161 | Duplicates Trashed: 55

Scanning and cleaning TRAIN set...


Class: Healthy_Nut: 100%|██████████| 1886/1886 [00:44<00:00, 42.81it/s]

✅ TRAIN Cleaned! Total: 8847 | Kept: 7965 | Duplicates Trashed: 882

🎉 Data Cleaning Complete! Your pure, un-leaked dataset is ready at: /kaggle/working/clean_arecanut


In [2]:
!pip install timm
!pip install brevitas
!pip install "numpy<2.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
import torch.nn.functional as F
import brevitas.nn as qnn

In [4]:
# 1. NEW CLEAN DATASET DEFINITIONS
class ArecanutDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data = ImageFolder(data_dir, transform=transform)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_transforms = transforms.Compose([
    transforms.Resize((64, 64)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
])

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# POINTING TO THE NEW CLEAN FOLDERS
train_folder = '/kaggle/working/clean_arecanut/train'
valid_folder = '/kaggle/working/clean_arecanut/test'

train_dataset = ArecanutDataset(train_folder, transform=train_transforms)
valid_dataset = ArecanutDataset(valid_folder, transform=transform)

print("Step 1: Multi-thread ripping CLEAN data to RAM...")

temp_train_loader = DataLoader(train_dataset, batch_size=128, num_workers=4)
temp_valid_loader = DataLoader(valid_dataset, batch_size=128, num_workers=4)

ram_train_images, ram_train_labels = [], []
for imgs, lbls in tqdm(temp_train_loader, desc="Caching Train"):
    ram_train_images.append(imgs)
    ram_train_labels.append(lbls)

ram_valid_images, ram_valid_labels = [], []
for imgs, lbls in tqdm(temp_valid_loader, desc="Caching Valid"):
    ram_valid_images.append(imgs)
    ram_valid_labels.append(lbls)

print("\nStep 2: Un-gluing batches to restore random shuffling...")
all_train_imgs = torch.cat(ram_train_images, dim=0)
all_train_lbls = torch.cat(ram_train_labels, dim=0)
all_valid_imgs = torch.cat(ram_valid_images, dim=0)
all_valid_lbls = torch.cat(ram_valid_labels, dim=0)

fast_train_dataset = TensorDataset(all_train_imgs, all_train_lbls)
fast_valid_dataset = TensorDataset(all_valid_imgs, all_valid_lbls)

train_loader = DataLoader(fast_train_dataset, batch_size=128, shuffle=True, pin_memory=True)
valid_loader = DataLoader(fast_valid_dataset, batch_size=128, shuffle=False, pin_memory=True)

print(f"✅ Data fully cached! Ready to train on {len(fast_train_dataset)} unique, clean images.")


# 2. THE 64-CHANNEL HARDWARE ARCHITECTURE
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=9):
        super(SimpleClassifier, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # LOCKED TO 64 CHANNELS 
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.global_pool = nn.AvgPool2d(kernel_size=8)
        
        self.conv_out = nn.Conv2d(64, num_classes, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        
        x = self.global_pool(x)
        
        x = self.conv_out(x)
        return x.view(x.size(0), -1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleClassifier(num_classes=9)
model.to(device)

num_epochs = 150
patience = 20  
patience_counter = 0

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)


# 3. DYNAMIC FOCAL LOSS (Automatically adjusts for deleted images)
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Automatically calculate exact class counts from the clean dataset
class_counts = np.bincount(train_dataset.data.targets)
total_samples = sum(class_counts)
num_classes = len(class_counts)

class_weights = [total_samples / (num_classes * count) for count in class_counts]
weights_tensor = torch.FloatTensor(class_weights).to(device)

criterion = FocalLoss(weight=weights_tensor, gamma=2.0)
print("✅ Dynamic Focal Loss initialized to match the newly cleaned dataset.")


# 4. Training Loop
best_val_loss = float('inf')
final_weights_name = 'arecanet_baseline_64_clean.pth'

print(f"\nStarting 96-Channel Baseline Training ({num_epochs} Epochs max)...")

val_acc_history = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1} Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    train_loss = running_loss / len(train_loader)

    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    valid_loss = running_loss / len(valid_loader)
    val_accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train loss: {train_loss:.4f}, Val loss: {valid_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")
    val_acc_history.append(val_accuracy)
    if valid_loss < best_val_loss:
        print(f"📉 Validation loss dropped! Saving pristine weights to {final_weights_name}")
        best_val_loss = valid_loss
        torch.save(model.state_dict(), final_weights_name)
        patience_counter = 0  
    else:
        patience_counter += 1
        print(f"⚠️ No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping triggered at Epoch {epoch+1}!")
            break

print("-" * 50)
print(f"✅ Training complete! Take '{final_weights_name}' to your Mixed-Precision Brevitas notebook.")

Step 1: Multi-thread ripping CLEAN data to RAM...


Caching Train:   0%|          | 0/63 [00:00<?, ?it/s]

Caching Valid:   0%|          | 0/17 [00:00<?, ?it/s]


Step 2: Un-gluing batches to restore random shuffling...
✅ Data fully cached! Ready to train on 7965 unique, clean images.
Using device: cuda
✅ Dynamic Focal Loss initialized to match the newly cleaned dataset.

Starting 96-Channel Baseline Training (150 Epochs max)...


Epoch 1 Training:   0%|          | 0/63 [00:00<?, ?it/s]

...

Epoch 148/150 - Train loss: 0.1422, Val loss: 0.2311, Val Accuracy: 80.70%
📉 Validation loss dropped! Saving pristine weights to arecanet_baseline_64_clean.pth


Epoch 149 Training:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 149/150 - Train loss: 0.1507, Val loss: 0.2433, Val Accuracy: 83.99%
⚠️ No improvement. Patience: 1/20


Epoch 150 Training:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 150/150 - Train loss: 0.1442, Val loss: 0.2817, Val Accuracy: 75.43%
⚠️ No improvement. Patience: 2/20
--------------------------------------------------
✅ Training complete! Take 'arecanet_baseline_64_clean.pth' to your Mixed-Precision Brevitas notebook.


In [5]:
import brevitas.nn as qnn

# 1. THE PURE 8-BIT, 64-CHANNEL ARCHITECTURE
class QuantArecaNet(nn.Module):
    def __init__(self, num_classes=9):
        super(QuantArecaNet, self).__init__()
        
        self.quant_inp = qnn.QuantIdentity(bit_width=8, return_quant_tensor=True)
        
        self.conv1 = qnn.QuantConv2d(3, 16, kernel_size=3, padding=1, weight_bit_width=8, bias=False)
        self.relu1 = qnn.QuantReLU(bit_width=8)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        
        self.conv2 = qnn.QuantConv2d(16, 32, kernel_size=3, padding=1, weight_bit_width=8, bias=False)
        self.relu2 = qnn.QuantReLU(bit_width=8)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) 

        self.conv3 = qnn.QuantConv2d(32, 64, kernel_size=3, padding=1, weight_bit_width=8, bias=False)
        self.relu3 = qnn.QuantReLU(bit_width=8)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) 
        
        self.global_pool = nn.AvgPool2d(kernel_size=8)
        
        self.conv_out = qnn.QuantConv2d(64, num_classes, kernel_size=1, bias=False, weight_bit_width=8)

    def forward(self, x):
        x = self.quant_inp(x)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        
        x = self.global_pool(x)
        
        x = self.conv_out(x)
        return x.view(x.size(0), -1)

q_model = QuantArecaNet(num_classes=9)

# 2. LOAD THE PERFECT FP32 BASELINE WEIGHTS
path_to_high_acc_weights = '/kaggle/working/arecanet_baseline_64_clean.pth' 

print("\nForce-loading the balanced baseline weights into 8-bit network...")
try:
    q_model.load_state_dict(torch.load(path_to_high_acc_weights, map_location=device), strict=False)
    print("✅ Baseline weights successfully loaded!")
except Exception as e:
    print(f"⚠️ Load Failed: {e}. Check if Cell 1 finished successfully!")

q_model.to(device)

# ==========================================
# 5. QUANTIZATION-AWARE TRAINING LOOP
# ==========================================
q_optimizer = optim.AdamW(q_model.parameters(), lr=0.0001, weight_decay=1e-4)

num_epochs = 150 
patience = 20
patience_counter = 0
best_val_loss = float('inf')
final_weights_path = 'quant_areca_weights_8BIT_64clean.pth'

print("\nStarting Pure 8-Bit QAT Pipeline...")

val1_acc_history = []

for epoch in range(num_epochs):
    q_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        q_optimizer.zero_grad()
        outputs = q_model(images)
        loss = criterion(outputs, labels) # Reusing FocalLoss from Cell 1!
        loss.backward()
        q_optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    q_model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = q_model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    valid_loss = running_loss / len(valid_loader)
    val_accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {valid_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")
    
    val1_acc_history.append(val_accuracy)

    if valid_loss < best_val_loss:
        print("📉 Validation loss dropped! Saving 8-bit weights...")
        best_val_loss = valid_loss
        torch.save(q_model.state_dict(), final_weights_path)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⚠️ No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping triggered at Epoch {epoch+1}!")
            break

print(f"\n🎉 8-Bit QAT Complete! Weights saved exclusively to: {final_weights_path}")
from IPython.display import FileLink
print("📦 Generating download link for your 8-bit weights...")
display(FileLink('quant_areca_weights_8BIT_64clean.pth'))


Force-loading the balanced baseline weights into 8-bit network...
✅ Baseline weights successfully loaded!

Starting Pure 8-Bit QAT Pipeline...


/usr/local/lib/python3.12/dist-packages/torch/_tensor.py:1679: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /pytorch/c10/core/TensorImpl.h:1973.)
  return super().rename(names)


Epoch 1/150 - Train Loss: 0.1149, Val Loss: 0.2268, Val Accuracy: 82.65%
📉 Validation loss dropped! Saving 8-bit weights...
Epoch 2/150 - Train Loss: 0.1136, Val Loss: 0.2229, Val Accuracy: 82.37%
📉 Validation loss dropped! Saving 8-bit weights...
Epoch 3/150 - Train Loss: 0.1106, Val Loss: 0.2332, Val Accuracy: 81.30%
⚠️ No improvement. Patience: 1/20
Epoch 4/150 - Train Loss: 0.1110, Val Loss: 0.2425, Val Accuracy: 80.93%
⚠️ No improvement. Patience: 2/20
Epoch 5/150 - Train Loss: 0.1108, Val Loss: 0.2210, Val Accuracy: 81.40%
📉 Validation loss dropped! Saving 8-bit weights...
Epoch 6/150 - Train Loss: 0.1113, Val Loss: 0.2302, Val Accuracy: 82.83%
⚠️ No improvement. Patience: 1/20
Epoch 7/150 - Train Loss: 0.1090, Val Loss: 0.2246, Val Accuracy: 81.77%
⚠️ No improvement. Patience: 2/20
Epoch 8/150 - Train Loss: 0.1155, Val Loss: 0.2277, Val Accuracy: 81.68%
⚠️ No improvement. Patience: 3/20
Epoch 9/150 - Train Loss: 0.1107, Val Loss: 0.2230, Val Accuracy: 82.23%
⚠️ No improvement. P

/kaggle/working/quant_areca_weights_8BIT_64clean.pth